# Vežba 8: Klasterizacija i PCA

k-means, hijerarhijska klasterizacija i DBSCAN, mere kvaliteta klastera, analiza glavnih komponenti i otkrivanje neobičnih dana

Lekcija sa objašnjenjima: [https://isatvss.surge.sh/vezba8.html](https://isatvss.surge.sh/vezba8.html)

Ćelije se izvršavaju redom, od vrha nadole (Shift+Enter), jer svaka koristi promenljive iz prethodnih.

## Algoritam k-means

### Zadatak 1. Priprema dana o kvalitetu vazduha

Učitati skup o vazduhu, zadržati dane u kojima postoje sva merenja i standardizovati deset obeležja.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

vazduh = pd.read_csv(
    "https://isatvss.surge.sh/podaci/vazduh_nis.csv", parse_dates=["datum"]
)
kolone = ["pm10", "pm25", "no2", "so2", "co", "o3"]
kolone += ["temperatura", "vetar_max", "padavine", "vlaznost"]
dani = vazduh.dropna(subset=kolone).reset_index(drop=True)
Z = StandardScaler().fit_transform(dani[kolone])
print(f"Dana sa svim merenjima: {len(dani)} od {len(vazduh)}")
print(f"Oblik standardizovane matrice: {Z.shape}")

### Zadatak 2. Tri tipa dana

Podeliti dane u tri klastera, ispisati broj dana i prosek svake veličine po klasteru, i raspodelu dana po mesecima.

Klasteri imaju jasno značenje. Klaster sa 333 dana su hladni dani sa jakim zagađenjem, PM10 prosečno 85,6 µg/m³ i temperaturom 4,7 °C, od oktobra do marta. Klaster sa 923 dana su topli dani sa visokim ozonom i malim zagađenjem česticama, od aprila do septembra. Treći klaster su vlažni dani sa više kiše (prosečno 3,9 mm) i umerenim zagađenjem, raspoređeni kroz celu godinu. Model nije znao za mesece, a sam je pronašao sezonu grejanja.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42).fit(Z)
dani["klaster"] = kmeans.labels_

print("Broj dana po klasteru:")
print(dani["klaster"].value_counts().sort_index())
print("\nProsek po klasteru, u originalnim jedinicama:")
print(dani.groupby("klaster")[kolone].mean().round(1).T)
print("\nUdeo dana po mesecima (redovi su klasteri):")
meseci = pd.crosstab(dani["klaster"], dani["datum"].dt.month, normalize="index")
print((meseci * 100).round(0).astype(int).to_string())

### Zadatak 3. Izbor broja klastera

Za k od 2 do 8 izračunati inerciju i silhouette koeficijent i nacrtati obe krive.

Silhouette je najveći za k = 2 (0,296), a za k = 3 iznosi 0,225. Lakat nije izražen. Mere, dakle, ne daju jednoznačan odgovor, što je kod stvarnih podataka čest slučaj, jer dani prelaze iz jednog tipa u drugi postepeno. Izbor k = 3 obrazložen je tumačenjem iz zadatka 2: treći klaster izdvaja kišne dane, što je smislena grupa. Takvo obrazloženje, uz vrednosti mera, ide u izveštaj.

In [ ]:
from sklearn.metrics import silhouette_score

broj_klastera = range(2, 9)
inercije, silueta = [], []
for k in broj_klastera:
    m = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
    inercije.append(m.inertia_)
    silueta.append(silhouette_score(Z, m.labels_))

fig, (levo, desno) = plt.subplots(1, 2, figsize=(11, 3.8))
levo.plot(broj_klastera, inercije, "-o", color="#5b3cc4")
levo.set_title("Metoda lakta: inercija")
levo.set_xlabel("Broj klastera k")
desno.plot(broj_klastera, silueta, "-o", color="#d9480f")
desno.set_title("Silhouette koeficijent")
desno.set_xlabel("Broj klastera k")
plt.tight_layout()
plt.show()
tabela = pd.DataFrame({"k": broj_klastera, "inercija": inercije, "silhouette": silueta})
print(tabela.round(3).to_string(index=False))

## Analiza glavnih komponenti

### Zadatak 4. Glavne komponente dana

Izračunati udeo rasipanja koji objašnjava svaka komponenta, doprinos obeležja prvim dvema komponentama, i nacrtati dane u ravni prve dve komponente sa bojom klastera.

Prve dve komponente objašnjavaju udeo 0,595 ukupnog rasipanja, a prvih pet 0,855. Prva komponenta ima velike pozitivne težine za čestice, ugljen-monoksid i azot-dioksid, a negativne za ozon i temperaturu, pa meri zimsko zagađenje naspram leta. Druga ima najveće težine za padavine i vlažnost. Na grafiku se tri klastera vide kao tri oblasti, a zimski dani se razvlače duž prve komponente.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(Z)
objasnjeno = pd.DataFrame(
    {
        "udeo": pca.explained_variance_ratio_,
        "kumulativno": pca.explained_variance_ratio_.cumsum(),
    },
    index=[f"PC{i + 1}" for i in range(len(kolone))],
)
print(objasnjeno.round(3).head(5))

opterecenja = pd.DataFrame(pca.components_[:2].T, index=kolone, columns=["PC1", "PC2"])
print("\nDoprinos obeležja prvim dvema komponentama:")
print(opterecenja.round(2))

P = pca.transform(Z)[:, :2]
plt.figure(figsize=(7, 5))
boje = np.array(["#5b3cc4", "#d9480f", "#2b8a3e"])
plt.scatter(P[:, 0], P[:, 1], c=boje[dani["klaster"]], s=8, alpha=0.6)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Dani u ravni prve dve glavne komponente, boja je klaster")
plt.show()

## Hijerarhijska klasterizacija

### Zadatak 5. Dendrogram i poređenje sa k-means

Podeliti dane u tri klastera Ward metodom, izmeriti slaganje sa k-means i nacrtati dendrogram za uzorak od 150 dana.

Slaganje sa k-means je 0,549: dva postupka se slažu u glavnim crtama, ali ne i u detaljima, posebno za dane na granici između tipova.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

hijerarhijska = AgglomerativeClustering(n_clusters=3, linkage="ward").fit(Z)
ari = adjusted_rand_score(kmeans.labels_, hijerarhijska.labels_)
print(f"Slaganje sa k-means (ARI): {ari:.3f}")

uzorak = np.random.default_rng(42).choice(len(Z), size=150, replace=False)
veze = linkage(Z[uzorak], method="ward")
plt.figure(figsize=(11, 4))
dendrogram(veze, no_labels=True, color_threshold=veze[-3, 2] + 0.01)
plt.title("Dendrogram za 150 nasumično izabranih dana (Ward)")
plt.ylabel("Rastojanje spajanja")
plt.show()

### Zadatak 6. Mere kvaliteta dve podele

Za podelu algoritmom k-means i hijerarhijsku podelu izračunati silhouette i Davies-Bouldin indeks.

Obe mere idu u prilog algoritmu k-means: silhouette 0,225 naspram 0,182 i Davies-Bouldin 1,604 naspram 1,788.

In [ ]:
from sklearn.metrics import davies_bouldin_score

for naziv, oznake in [
    ("k-means", kmeans.labels_),
    ("hijerarhijska", hijerarhijska.labels_),
]:
    print(f"{naziv:14s} silhouette = {silhouette_score(Z, oznake):.3f}", end="")
    print(f", Davies-Bouldin = {davies_bouldin_score(Z, oznake):.3f}")

## DBSCAN: žarišta nezgoda

### Zadatak 7. Žarišta nezgoda u Nišu

Na nezgodama sa teritorije grada Niša primeniti DBSCAN sa rastojanjem po površini Zemlje, poluprečnikom 150 m i bar osam nezgoda u okolini, pa ispisati broj žarišta, udeo šuma i pet najvećih žarišta.

Na 919 nezgoda pronađeno je 15 žarišta, a 70,1 % nezgoda ne pripada nijednom, jer su raspršene po gradu. Najveće žarište ima 48 nezgoda, a udeo teških u njemu je 0,583.

In [ ]:
from sklearn.cluster import DBSCAN

nezgode = pd.read_csv("https://isatvss.surge.sh/podaci/nezgode_2025.csv")
nezgode = nezgode.drop_duplicates(subset="id_nezgode")
nezgode["teska"] = (nezgode["posledica"] != "Sa mat.stetom").astype(int)
nis = nezgode[nezgode["opstina"].isin(["NIŠ", "NIŠKA BANJA"])]
nis = nis[nis["lat"].between(43.2, 43.45) & nis["lon"].between(21.7, 22.2)].copy()

poluprecnik_zemlje_km = 6371
radijani = np.radians(nis[["lat", "lon"]].to_numpy())
dbscan = DBSCAN(
    eps=0.15 / poluprecnik_zemlje_km, min_samples=8, metric="haversine"
).fit(radijani)
nis["klaster"] = dbscan.labels_

print(f"Nezgoda na teritoriji grada: {len(nis)}")
print(f"Pronađeno žarišta: {nis['klaster'].max() + 1}")
print(f"Nezgoda van žarišta (šum): {(nis['klaster'] == -1).mean():.1%}")

zarista = (
    nis[nis["klaster"] >= 0]
    .groupby("klaster")
    .agg(
        nezgoda=("teska", "size"),
        udeo_teskih=("teska", "mean"),
        lat=("lat", "mean"),
        lon=("lon", "mean"),
    )
    .sort_values("nezgoda", ascending=False)
)
print("\nPet najvećih žarišta:")
print(zarista.head().round(3).to_string())

### Zadatak 8. Mapa žarišta i poređenje sa k-means

Nacrtati žarišta pronađena algoritmom DBSCAN i podelu iste teritorije algoritmom k-means sa istim brojem klastera.

DBSCAN pronalazi izdužena žarišta koja prate ulice, a raštrkane nezgode ostavlja kao šum. K-means mora da rasporedi svaku nezgodu, pa deli celu teritoriju na približno okrugle oblasti, i one nisu žarišta nego zone. Za pitanje „gde se nezgode gomilaju“ odgovarajući alat je DBSCAN.

In [ ]:
fig, (levo, desno) = plt.subplots(1, 2, figsize=(13, 5.5))
sum_tacke = nis["klaster"] == -1
levo.scatter(nis.loc[sum_tacke, "lon"], nis.loc[sum_tacke, "lat"], s=8, color="#c0c0c0")
levo.scatter(
    nis.loc[~sum_tacke, "lon"],
    nis.loc[~sum_tacke, "lat"],
    c=nis.loc[~sum_tacke, "klaster"],
    cmap="tab20",
    s=18,
)
levo.set_xlim(21.86, 21.96)
levo.set_ylim(43.295, 43.345)
levo.set_title("DBSCAN, uže gradsko područje: žarišta i šum (sivo)")

km_nis = KMeans(n_clusters=nis["klaster"].max() + 1, n_init=10, random_state=42)
km_oznake = km_nis.fit_predict(nis[["lon", "lat"]])
desno.scatter(nis["lon"], nis["lat"], c=km_oznake, cmap="tab20", s=6)
desno.set_title("k-means sa istim brojem klastera, cela teritorija grada")
for osa in (levo, desno):
    osa.set_xlabel("Geografska dužina")
    osa.set_ylabel("Geografska širina")
    osa.set_aspect(1 / np.cos(np.radians(43.3)))
plt.tight_layout()
plt.show()

## Otkrivanje neobičnih dana

### Zadatak 9. Isolation Forest

Izdvojiti 1 % najneobičnijih dana metodom Isolation Forest i ispisati šest najneobičnijih.

Najneobičniji dani su januarske epizode ekstremnog zagađenja, sa PM10 do 195,2 µg/m³ 18. januara 2019. To nisu greške merenja, nego stvarni i najvažniji događaji u skupu, kao na vežbi 2. Otkrivanje outliera ovde ne služi za brisanje, nego za pronalaženje dana koje vredi posebno proučiti.

In [ ]:
from sklearn.ensemble import IsolationForest

izolacija = IsolationForest(contamination=0.01, random_state=42).fit(Z)
dani["anomalija"] = izolacija.predict(Z) == -1
dani["skor"] = izolacija.score_samples(Z)
print(f"Označeno neobičnih dana: {dani['anomalija'].sum()}")
najneobicniji = dani.sort_values("skor").head(6)
print(najneobicniji[["datum"] + kolone].round(1).to_string(index=False))

## Zadaci za samostalan rad

### Zadatak 10. k-means ručno

Tačke 1, 2, 3, 8, 9 i 15 na brojevnoj pravoj podeliti u dva klastera, sa početnim centrima 1 i 2. Ručno izvesti dodelu i pomeranje centara do stabilnog stanja, pa rezultat proveriti kodom.

### Zadatak 11. k-means bez skaliranja

Ponoviti k-means sa tri klastera nad nestandardizovanim podacima o vazduhu i izmeriti slaganje sa podelom iz zadatka 2. Koja obeležja najviše utiču na nestandardizovanu podelu?

### Zadatak 12. Uticaj poluprečnika eps

Ponoviti DBSCAN nad nezgodama u Nišu za eps od 0,1, 0,15, 0,3 i 0,5 km i uporediti broj žarišta i udeo šuma. Kakva žarišta daje premali, a kakva preveliki eps?

### Zadatak 13. k-means nad PCA komponentama

Primeniti k-means sa tri klastera nad prvih 2, 5 i 10 glavnih komponenti i prilagođenim Rand indeksom uporediti rezultat sa podelom nad svim obeležjima.

### Zadatak 14. LOF i Isolation Forest

Izdvojiti 1 % neobičnih dana metodom Local Outlier Factor i uporediti ih sa danima iz zadatka 9. Koliko dana označavaju oba postupka?